# 01. Línea base secuencial (Keras)

Código de entrenamiento del lab 3 (`02_deep_learning.ipynb`) sin análisis ni gráficas.
Es la **solución secuencial original** contra la que se mide el speedup.

Dos fases separadas y cronometradas por separado:
1. **Preparación de datos (CPU):** leer JPEG, decodificar, redimensionar, normalizar.
2. **Entrenamiento:** forward/backward de la red.

Poner `FORCE_CPU = True` para la medición secuencial. Con `False` usa la GPU y sirve como
referencia de "Keras en GPU".

In [ ]:
import os

FORCE_CPU = True   # True: línea base secuencial. False: Keras sobre GPU.
if FORCE_CPU:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'   # debe fijarse antes de importar TensorFlow

import sys, json, time
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import f1_score

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))
from asl.data import (load_splits, get_tf_datasets, CLASS_NAMES, NUM_CLASSES,
                      IMG_SIZE, RANDOM_SEED)

keras.utils.set_random_seed(RANDOM_SEED)

RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

INPUT_SHAPE = (IMG_SIZE[0], IMG_SIZE[1], 3)
BATCH_SIZE = 64
DEVICE = 'gpu' if tf.config.list_physical_devices('GPU') else 'cpu'

# Topes de épocas del lab 3. El early stopping suele cortar antes.
EPOCHS = {'CNN-1 Baseline': 20, 'CNN-2 Profunda': 25, 'CNN-3 Regularizada': 25}

print(f'TensorFlow {tf.__version__}, dispositivo: {DEVICE}, núcleos CPU: {os.cpu_count()}')

## Fase 1. Preparación de datos (CPU)

In [ ]:
t0 = time.perf_counter()
df_train, df_val, df_test = load_splits()
ds_train, ds_val, ds_test = get_tf_datasets(
    df_train, df_val, df_test,
    img_size=IMG_SIZE, batch_size=BATCH_SIZE, num_classes=NUM_CLASSES,
)

# Los datasets son perezosos. Recorrerlos una vez fuerza la lectura y decodificación de todos
# los JPEG y llena el cache(), así este tiempo queda aparte del de entrenamiento.
for ds in (ds_train, ds_val, ds_test):
    for _ in ds:
        pass
t_data_prep = time.perf_counter() - t0

y_test = df_test['label_id'].values
print(f'Preparación de datos: {t_data_prep:.1f} s')

## Modelos (idénticos a los del lab 3)

In [ ]:
def build_cnn1():
    return keras.Sequential([
        keras.Input(shape=INPUT_SHAPE),
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name='CNN_1_Baseline')


def build_cnn2():
    inputs = keras.Input(shape=INPUT_SHAPE)
    x = inputs
    for filtros in (64, 128, 256):
        x = layers.Conv2D(filtros, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return keras.Model(inputs, outputs, name='CNN_2_Profunda')


def build_cnn3():
    inputs = keras.Input(shape=INPUT_SHAPE)
    x = layers.Conv2D(64, 5, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.SpatialDropout2D(0.1)(x)

    for filtros in (128, 256):
        x = layers.Conv2D(filtros, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.SpatialDropout2D(0.1)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return keras.Model(inputs, outputs, name='CNN_3_Regularizada')


BUILDERS = {'CNN-1 Baseline': build_cnn1, 'CNN-2 Profunda': build_cnn2,
            'CNN-3 Regularizada': build_cnn3}

## Fase 2. Entrenamiento

In [ ]:
class EpochTimer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None): self.times = []
    def on_epoch_begin(self, epoch, logs=None): self._t = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None): self.times.append(time.perf_counter() - self._t)


def entrenar(model, epochs, lr=1e-3, paciencia=5, early_stopping=True):
    """early_stopping=False corre exactamente `epochs` épocas, útil para comparar tiempos."""
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    timer = EpochTimer()
    callbacks = [timer, keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6)]
    if early_stopping:
        callbacks.append(keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=paciencia, restore_best_weights=True))

    t0 = time.perf_counter()
    history = model.fit(ds_train, validation_data=ds_val, epochs=epochs,
                        callbacks=callbacks, verbose=2)
    return history, time.perf_counter() - t0, timer.times


def evaluar(model):
    y_pred = np.argmax(model.predict(ds_test, verbose=0), axis=1)
    return (float((y_pred == y_test).mean()),
            float(f1_score(y_test, y_pred, average='macro')))

In [ ]:
EARLY_STOPPING = True    # False para la medición de tiempos con épocas fijas
EPOCHS_OVERRIDE = None   # p. ej. 5 para una corrida corta de tiempos

resultados = {'device': DEVICE, 'cpu_cores': os.cpu_count(),
              'data_prep_seconds': round(t_data_prep, 2), 'modelos': {}}

for nombre, build in BUILDERS.items():
    keras.utils.set_random_seed(RANDOM_SEED)
    model = build()
    epochs = EPOCHS_OVERRIDE or EPOCHS[nombre]
    print(f'\n=== {nombre}, {model.count_params():,} parámetros ===')

    history, t_train, t_epochs = entrenar(model, epochs, early_stopping=EARLY_STOPPING)
    acc, f1 = evaluar(model)

    resultados['modelos'][nombre] = {
        'parametros': int(model.count_params()),
        'epocas': len(t_epochs),
        'train_seconds': round(t_train, 2),
        'epoch_seconds': [round(t, 2) for t in t_epochs],
        'accuracy': acc, 'f1_macro': f1,
    }
    print(f'{nombre}: {acc*100:.2f} % accuracy, F1 {f1:.4f}, {t_train:.0f} s en {len(t_epochs)} épocas')

In [ ]:
salida = RESULTS_DIR / f'keras_{DEVICE}.json'
with open(salida, 'w', encoding='utf-8') as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)
print(f'Resultados en {salida}')